# Load data
Trong phần này, thứ ta cần làm chính là chuyển 2 file data raw thành 1 df thống nhất

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/raw")


PROJECT_ROOT = find_project_root()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
df_shoppee = pd.read_json(
    DATA_RAW_DIR / "shopee_reviews_dataset.jsonl",
    lines=True
)
df_shoppee.head(), df_shoppee.shape

(            id                                             review  rating  \
 0  74263765409  Hương vị:thom  Chắc do bên giao hàng bị vỡ mấy...       3   
 1  11104151002  Hương thơm:nhẹ nhàng Lợi ích:phục hồi cấp ẩm M...       5   
 2  15888299382  Chất lượng sản phẩm:ok Đúng với mô tả:đúng  Lầ...       5   
 3  81030214453  Độ tuổi sử dụng:em bes Chất lượng sản phẩm:tot...       5   
 4  88484377297  Hương vị:Mix vị, Tím  Mình mua 64 gói (32 gói ...       1   
 
       label  
 0  negative  
 1  positive  
 2  positive  
 3  positive  
 4  negative  ,
 (9599, 4))

In [5]:
df_aug = pd.read_json(
    DATA_RAW_DIR / "aug_unaccented_reviews.jsonl",
    lines=True
)
df_aug.head(), df_aug.shape

(            id                                             review  rating  \
 0  59241444271  Mui huong:Ngot nhe, thomm Phu hop voi loai da:...       4   
 1  75292434640  Toi rat hai long voi dich vu cua nguoi ban va ...       5   
 2  13364343551  Review nhe mot so dong sua con uong tang can t...       5   
 3  84816190204  Hieu qua:Sach Thiet ke:Dep  Thiet ke nho gon, ...       5   
 4  30102564756  Giao hang nhanh, dong goi dep an toan Mui Pink...       4   
 
       label  
 0  positive  
 1  positive  
 2  positive  
 3  positive  
 4  positive  ,
 (1348, 4))

In [6]:
df_all = pd.concat([df_aug, df_shoppee], ignore_index=True)
df_all.shape

(10947, 4)

# Clean and Normalize text
Ta cần nhớ yêu cầu của 2 mô hình sẽ khác nhau:
- Đối với mô hình LSTM thì yêu cầu đơn giản
- Đối với mô hình baseline cần xử lí kỹ và phức tạp hớn

### Clean
Trong phần clean này, những điều cần làm chung cho cả hai mô hình:
- Bỏ url, html, email
- Chuyển emoji thành text
- Chuẩn hóa khoảng trắng

Riêng đối mới mô hình cho baseline cần xử lí một số thứ đặt biệt hơn
- Lower
- Bỏ dấu câu



In [7]:
import re

EMOJI_SENTIMENT_MAP = {
    # Tích cực
    "😍": "positive_emoji",
    "🥰": "positive_emoji",
    "😘": "positive_emoji",
    "❤️": "positive_emoji",
    "💖": "positive_emoji",
    "💕": "positive_emoji",
    "💝": "positive_emoji",
    "😊": "positive_emoji",
    "😁": "positive_emoji",
    "😄": "positive_emoji",
    "😃": "positive_emoji",
    "😀": "positive_emoji",
    "🙂": "positive_emoji",
    "👍": "positive_emoji",
    "👏": "positive_emoji",
    "✨": "positive_emoji",
    "🌟": "positive_emoji",
    "🔥": "positive_emoji",
    "🎉": "positive_emoji",

    # Tiêu cực
    "😡": "negative_emoji",
    "😠": "negative_emoji",
    "😤": "negative_emoji",
    "😞": "negative_emoji",
    "😔": "negative_emoji",
    "😢": "negative_emoji",
    "😭": "negative_emoji",
    "😥": "negative_emoji",
    "☹️": "negative_emoji",
    "🙁": "negative_emoji",
    "👎": "negative_emoji",
    "💔": "negative_emoji",
    "🤬": "negative_emoji",

    # Hài hước
    "😂": "laugh_emoji",
    "🤣": "laugh_emoji",
    "😆": "laugh_emoji",

    # Bất ngờ
    "😮": "surprise_emoji",
    "😲": "surprise_emoji",
    "😱": "surprise_emoji",

    # Trung tính
    "😐": "neutral_emoji",
    "😑": "neutral_emoji",
    "🤔": "neutral_emoji",

    # Chê bai / mỉa mai
    "🙄": "sarcasm_emoji",
    "😒": "sarcasm_emoji",

    # Tức giận mạnh
    "🤯": "angry_emoji",
    "💢": "angry_emoji",
}

EMOJI_PATTERN = re.compile(
    "|".join(re.escape(emoji) for emoji in EMOJI_SENTIMENT_MAP.keys())
)

def replace_emoji_sentiment(text: str) -> str:
    if not isinstance(text, str):
        return ""

    return EMOJI_PATTERN.sub(
        lambda m: f" {EMOJI_SENTIMENT_MAP[m.group(0)]} ",
        text
    )

def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

def remove_html(text):
    return re.sub(r"<.*?>", "", text)

def remove_email(text: str) -> str:
    if not isinstance(text, str):
        return ""
    email_pattern = r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b'
    return re.sub(email_pattern, '', text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()
    # turn consecutive whitespaces into one, remove begin/end whitespace

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = remove_url(text)
    text = remove_html(text)
    text = normalize_whitespace(text)
    text = replace_emoji_sentiment(text)
    return text

In [8]:
#apply cho cả 2 dataset
df_baseline = df_all.copy()
df_lstm = df_all.copy()

df_baseline["review"] = df_baseline["review"].apply(clean_text)
df_lstm["review"] = df_lstm["review"].apply(clean_text)

df_lstm.head(), df_baseline.head()

(            id                                             review  rating  \
 0  59241444271  mui huong:ngot nhe, thomm phu hop voi loai da:...       4   
 1  75292434640  toi rat hai long voi dich vu cua nguoi ban va ...       5   
 2  13364343551  review nhe mot so dong sua con uong tang can t...       5   
 3  84816190204  hieu qua:sach thiet ke:dep thiet ke nho gon, t...       5   
 4  30102564756  giao hang nhanh, dong goi dep an toan mui pink...       4   
 
       label  
 0  positive  
 1  positive  
 2  positive  
 3  positive  
 4  positive  ,
             id                                             review  rating  \
 0  59241444271  mui huong:ngot nhe, thomm phu hop voi loai da:...       4   
 1  75292434640  toi rat hai long voi dich vu cua nguoi ban va ...       5   
 2  13364343551  review nhe mot so dong sua con uong tang can t...       5   
 3  84816190204  hieu qua:sach thiet ke:dep thiet ke nho gon, t...       5   
 4  30102564756  giao hang nhanh, dong goi dep an

In [9]:
# Xử lí thêm cho baseline
def remove_punctuation(text):
    return re.sub(r"[^\w\s]", " ", text)

df_baseline["review"] = df_baseline["review"].apply(remove_punctuation)
df_baseline["review"] = df_baseline["review"].str.lower()

df_baseline.head()


,id,review,rating,label
0,59241444271,mui huong ngot nhe thomm phu hop voi loai da ...,4,positive
1,75292434640,toi rat hai long voi dich vu cua nguoi ban va ...,5,positive
2,13364343551,review nhe mot so dong sua con uong tang can t...,5,positive
3,84816190204,hieu qua sach thiet ke dep thiet ke nho gon t...,5,positive
4,30102564756,giao hang nhanh dong goi dep an toan mui pink...,4,positive


# Normalize_text
Trong phần này ta cần:
- Chuyển tiếng Việt không dấu -> có dấu
- Đổi lại các từ viết tắt
- Chuẩn hóa lại các từ bị kéo dài

Riêng với xử lí cho baseline: cần thêm bỏ stopword

In [10]:
import re
# Loại bỏ từ bị kéo dài (lặp lại từ 3 chữ cái trở lên)
def normalize_elongated_words(text: str) -> str:
    if not isinstance(text, str):
        return ""

    return re.sub(r'(.)\1{2,}', r'\1\1', text)



In [11]:
import unicodedata
from collections import Counter, defaultdict

SLANG_DICT = {
    "ko": "không",
    "khum": "không",
    "hok": "không",
    "k": "không",
    "kh": "không",
    "dc": "được",
    "đc": "được",
    "mn": "mọi người",
    "ntn": "như thế nào",
    "n": "nhiều",
    "sp": "sản phẩm",
    "shop": "shop",
    "ship": "ship",
    "shipper": "shipper",
    "siu": "siêu",
    "l": "lần",
    "vs": "với"
}

DIACRITIC_OVERRIDES = {
    "a": "à",
    "ai": "ai",
    "anh": "ảnh",
    "ban": "bạn",
    "bao": "bao",
    "bi": "bị",
    "biet": "biết",
    "binh": "bình",
    "bo": "bỏ",
    "cam": "cảm",
    "can": "cần",
    "cao": "cao",
    "chat": "chất",
    "che": "chê",
    "chi": "chỉ",
    "cho": "cho",
    "chong": "chống",
    "chua": "chưa",
    "co": "có",
    "con": "còn",
    "cua": "của",
    "cung": "cũng",
    "da": "da",
    "danh": "đánh",
    "de": "dễ",
    "den": "đến",
    "dep": "đẹp",
    "di": "đi",
    "dich": "dịch",
    "dieu": "điều",
    "dong": "đóng",
    "duoc": "được",
    "gia": "giá",
    "giao": "giao",
    "giup": "giúp",
    "hang": "hàng",
    "hay": "hay",
    "hieu": "hiệu",
    "hoi": "hơi",
    "hon": "hơn",
    "huong": "hương",
    "khong": "không",
    "la": "là",
    "lai": "lại",
    "lam": "lắm",
    "lan": "lần",
    "loai": "loại",
    "luong": "lượng",
    "minh": "mình",
    "moi": "mới",
    "mot": "một",
    "mua": "mua",
    "muc": "mức",
    "mui": "mùi",
    "muon": "muốn",
    "nay": "này",
    "nen": "nên",
    "ngot": "ngọt",
    "ngay": "ngày",
    "nghi": "nghĩ",
    "nguoi": "người",
    "nhanh": "nhanh",
    "nhat": "nhất",
    "nho": "nhỏ",
    "nhieu": "nhiều",
    "nhin": "nhìn",
    "nhung": "nhưng",
    "noi": "nói",
    "qua": "quá",
    "rat": "rất",
    "roi": "rồi",
    "san": "sản",
    "sao": "sao",
    "se": "sẽ",
    "sop": "shop",
    "su": "sự",
    "tam": "tạm",
    "thay": "thấy",
    "them": "thêm",
    "thi": "thì",
    "thich": "thích",
    "tot": "tốt",
    "toi": "tôi",
    "trai": "trải",
    "trang": "trắng",
    "truoc": "trước",
    "tuyet": "tuyệt",
    "va": "và",
    "van": "vẫn",
    "ve": "về",
    "vi": "vì",
    "voi": "với",
}

PHRASE_OVERRIDES = {
    "chat luong": "chất lượng",
    "cham soc": "chăm sóc",
    "chong nang": "chống nắng",
    "dich vu": "dịch vụ",
    "dong goi": "đóng gói",
    "dong sua": "dòng sữa",
    "don hang": "đơn hàng",
    "giai dap": "giải đáp",
    "giao hang": "giao hàng",
    "hai long": "hài lòng",
    "ho tro": "hỗ trợ",
    "khach hang": "khách hàng",
    "khong bi": "không bị",
    "loai da": "loại da",
    "moi nguoi": "mọi người",
    "mot so": "một số",
    "nguoi ban": "người bán",
    "phu hop": "phù hợp",
    "qua trinh": "quá trình",
    "san pham": "sản phẩm",
    "su chuyen nghiep": "sự chuyên nghiệp",
    "tang can": "tăng cân",
    "thai do": "thái độ",
    "thoi gian": "thời gian",
    "trai nghiem": "trải nghiệm",
    "van chuyen": "vận chuyển",
    "kcn": "kem chống nắng"
}

def strip_vietnamese_accents(text):
    text = unicodedata.normalize("NFD", str(text))
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("đ", "d").replace("Đ", "D")

def build_diacritic_restore_map(texts):
    candidates = defaultdict(Counter)
    for text in texts.dropna().astype(str):
        for token in re.findall(r"\w+", text.lower(), flags=re.UNICODE):
            key = strip_vietnamese_accents(token)
            if key:
                candidates[key][token] += 1

    restore_map = {
        key: counter.most_common(1)[0][0]
        for key, counter in candidates.items()
    }
    restore_map.update(DIACRITIC_OVERRIDES)
    restore_map.update(SLANG_DICT)
    return restore_map

def preserve_case(original, restored):
    if original.isupper():
        return restored.upper()
    if original[:1].isupper():
        return restored.capitalize()
    return restored

def has_vietnamese_diacritics(token):
    return token != strip_vietnamese_accents(token)

def apply_phrase_overrides(text):
    restored = str(text)
    for phrase, replacement in sorted(PHRASE_OVERRIDES.items(), key=lambda item: len(item[0]), reverse=True):
        restored = re.sub(rf"\b{re.escape(phrase)}\b", replacement, restored)
    return restored

def restore_vietnamese_diacritics(text, restore_map):
    text = apply_phrase_overrides(text)
    tokens = re.findall(r"\w+|\s+|[^\w\s]", str(text), flags=re.UNICODE)
    restored_tokens = []
    for token in tokens:
        if re.fullmatch(r"\w+", token, flags=re.UNICODE):
            if has_vietnamese_diacritics(token.lower()):
                restored_tokens.append(token)
                continue
            key = strip_vietnamese_accents(token.lower())
            restored = restore_map.get(key, token)
            restored_tokens.append(preserve_case(token, restored))
        else:
            restored_tokens.append(token)
    return "".join(restored_tokens)

def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

def normalize_slang(text):
    words = text.split()
    norm_words = []
    for word in words:
        norm_words.append(SLANG_DICT.get(word, word))
        # replace if it's a slang
    return " ".join(norm_words)

def normalize_repeated_characters(text):
    # Reduce expressive 3+ repeated chars, but keep valid doubles like "shipper".
    return re.sub(r"(.)\1{2,}", r"\1", text)

def normalize_text(text):
    text = normalize_unicode(text)
    text = normalize_slang(text)
    text = normalize_repeated_characters(text)
    text = normalize_whitespace(text)

    return text


In [12]:
import pandas as pd

all_reviews = pd.concat(
    [df_baseline["review"], df_lstm["review"]],
    ignore_index=True
)

accent_restore_map = build_diacritic_restore_map(all_reviews)

print(
    f"Accent restoration dictionary size: "
    f"{len(accent_restore_map):,} tokens"
)

Accent restoration dictionary size: 4,281 tokens


In [13]:
df_baseline["review"] = (
    df_baseline["review"].apply(normalize_text)
)

df_lstm["review"] = (
    df_lstm["review"].apply(normalize_text)
)

In [14]:
def preprocess_df(df, restore_map):
    df = df.copy()

    df["review"] = (
        df["review"]
        .fillna("")
        .astype(str)
        .apply(
            lambda x: restore_vietnamese_diacritics(
                x,
                restore_map
            )
        )
        .apply(normalize_text)
    )

    return df

In [15]:
accent_restore_map = build_diacritic_restore_map(
    pd.concat(
        [
            df_baseline["review"],
            df_lstm["review"]
        ],
        ignore_index=True
    )
)

df_baseline = preprocess_df(
    df_baseline,
    accent_restore_map
)

df_lstm = preprocess_df(
    df_lstm,
    accent_restore_map
)

In [16]:
# apply cho cả hai datadframe


df_baseline.head(30)

,id,review,rating,label
0,59241444271,mùi hương ngọt nhẹ thomm phù hợp với loại da m...,4,positive
1,75292434640,tôi rất hài lòng với dịch vụ của người bán và ...,5,positive
2,13364343551,review nhẹ một số dòng sữa còn uống tăng cân t...,5,positive
3,84816190204,hiệu quá sạch thiết kế đẹp thiết kế nhỏ gọn ti...,5,positive
4,30102564756,giao hàng nhanh đóng gói đẹp ăn toàn mùi pink ...,4,positive
5,15901897362,doi tưởng sự dụng everyone chất lượng sản phẩm...,5,positive
6,81495231201,ẩm thanh chất lượng ổn ẩm thanh nghe khá ổn ba...,4,positive
7,41936001824,mùi hương pink heaven do lưu hương 2 3h um cả ...,5,positive
8,77435108286,i m really impressed with this product thế qua...,5,positive
9,80025894094,lợi ích sản phẩm dụng rất ok nha dụng da đều m...,5,positive


In [17]:
# Bỏ stopword với các stopword được định nghĩa trong "vietnamese-stopwords.txt"
import os

def load_stopwords(filepath):
    """
    Học/Đọc danh sách stopwords từ file text.
    Mỗi dòng trong file là một stopword.
    """
    stopwords = set()
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                word = line.strip()
                if word:
                    # Chuyển về chữ thường để khớp chính xác khi so sánh
                    stopwords.add(word.lower())
                    # Nếu dữ liệu đã tách từ dạng 'sản_phẩm', hãy đảm bảo stopword cũng dùng dấu gạch dưới
                    # Hàm này tự động hỗ trợ cả khoảng trắng lẫn dấu gạch dưới
                    stopwords.add(word.replace(" ", "_"))
        print(f"-> Đã tải thành công {len(stopwords)} từ dừng (gồm cả biến thể gạch dưới).")
    else:
        print(f"⚠️ Cảnh báo: Không tìm thấy file stopword tại đường dẫn: {filepath}")
    return stopwords


def remove_stopwords(text, stopwords_set):
    """
    Hàm xóa bỏ các stopwords khỏi một văn bản chuỗi (text).
    """
    # Kiểm tra nếu dữ liệu khuyết thiếu (NaN/Null)
    if not isinstance(text, str) or not text.strip():
        return ""
    
    # Tách chuỗi thành danh sách các từ dựa trên khoảng trắng
    words = text.split()
    
    # Giữ lại các từ KHÔNG nằm trong danh sách stopwords
    # Dùng list comprehension để tối ưu tốc độ
    filtered_words = [word for word in words if word.lower() not in stopwords_set]
    
    # Gộp các từ còn lại thành chuỗi hoàn chỉnh
    return " ".join(filtered_words)


In [19]:
# Bỏ stopword cho df baseline
stopwords_path = PROJECT_ROOT / "data" / "vietnamese-stopwords.txt"


vietnamese_stopwords = load_stopwords(stopwords_path)
df_baseline["review"] = df_baseline["review"].apply(lambda x: remove_stopwords(x, vietnamese_stopwords))
df_baseline.head()

-> Đã tải thành công 3513 từ dừng (gồm cả biến thể gạch dưới).


,id,review,rating,label
0,59241444271,mùi hương nhẹ thomm phù hợp da da công dụng dư...,4,positive
1,75292434640,hài dịch vụ shipper sản phẩm mua chất tuyệt gi...,5,positive
2,13364343551,review nhẹ dòng sữa uống cân cm quan tạm grow ...,5,positive
3,84816190204,hiệu sạch thiết kế đẹp thiết kế gọn tiền lợi k...,5,positive
4,30102564756,giao hàng đóng gói đẹp toàn mùi pink review da...,4,positive


# Word_segmentation

Trong phần này, vì đặc thù của chữ Tiếng Việt, một vài từ phải đi cùng nhau mới có ý nghĩa được, như việc em đi cùng anh mới có ý nghĩa. Nên ta phải gom các từ cần thiết lại với nhau, ví dụ hoc sinh -> hoc_sinh


In [20]:
from underthesea import word_tokenize
from pyvi.ViTokenizer import tokenize

def segment_underthesea(text):
    return word_tokenize(text, format="text")
    # segment and return a string (format="text")

def segment_pyvi(text):
    return tokenize(text)

def segment_text(text, method="underthesea"):
    if method == "underthesea":
        return segment_underthesea(text)
    elif method == "pyvi":
        return segment_pyvi(text)
    else:
        raise ValueError("Invalid segmentation method")

In [ ]:
import os
import pandas as pd

# 1. Đảm bảo thư mục 'data/interim' đã tồn tại (tránh lỗi nếu chưa có thư mục)
output_dir = DATA_INTERIM_DIR
os.makedirs(output_dir, exist_ok=True)

# Giả sử cột chứa văn bản cần tách từ trong df của bạn tên là "review"
# Bạn có thể đổi tên cột này cho đúng với thực tế dữ liệu của nhóm nhé.
target_column = "review"

# 2. Tiến hành tách từ (Word Segmentation) bằng hàm segment_text đã viết trước đó
print("Đang tách từ cho df_baseline...")
# Ví dụ: Dùng PyVi cho mô hình Baseline để xử lý nhanh
df_baseline["segmented_review"] = df_baseline[target_column].apply(
    lambda x: segment_text(str(x), method="pyvi")
)

print("Đang tách từ cho df_lstm...")
# Ví dụ: Dùng Underthesea cho mô hình LSTM để ngữ nghĩa chính xác hơn, hoặc đổi thành "pyvi" nếu muốn đồng nhất
df_lstm["segmented_review"] = df_lstm[target_column].apply(
    lambda x: segment_text(str(x), method="underthesea")
)

# 3. Lưu dữ liệu ra file CSV trong folder interim
# Thiết lập index=False để khi lưu không bị sinh thêm một cột số thứ tự không mong muốn
baseline_path = os.path.join(output_dir, "df_baseline_segmented.csv")
lstm_path = os.path.join(output_dir, "df_lstm_segmented.csv")

df_baseline.to_csv(baseline_path, index=False, encoding="utf-8-sig")
df_lstm.to_csv(lstm_path, index=False, encoding="utf-8-sig")

print(f"Lưu file thành công tại:\n - {baseline_path}\n - {lstm_path}")

<>:5: SyntaxWarning: invalid escape sequence '\T'
<>:5: SyntaxWarning: invalid escape sequence '\T'
C:\Users\Tony\AppData\Local\Temp\ipykernel_6212\914448053.py:5: SyntaxWarning: invalid escape sequence '\T'
  output_dir = "D:\Tony\Career\ProjectNhom\Customer_sentiment_analysis\data\interim"


Đang tách từ cho df_baseline...
Đang tách từ cho df_lstm...
Lưu file thành công tại:
 - D:\Tony\Career\ProjectNhom\Customer_sentiment_analysis\data\interim\df_baseline_segmented.csv
 - D:\Tony\Career\ProjectNhom\Customer_sentiment_analysis\data\interim\df_lstm_segmented.csv


In [21]:
df_lstm.head()

,id,review,rating,label
0,59241444271,"mùi hương:ngọt nhẹ, thomm phù hợp với loại da:...",4,positive
1,75292434640,tôi rất hài lòng với dịch vụ của người bán và ...,5,positive
2,13364343551,review nhẹ một số dòng sữa còn uống tăng cân t...,5,positive
3,84816190204,"hiệu quá:sạch thiết kế:đẹp thiết kế nhỏ gọn, t...",5,positive
4,30102564756,"giao hàng nhanh, đóng gói đẹp ăn toàn mùi pink...",4,positive


In [22]:
df_baseline.head()

,id,review,rating,label
0,59241444271,mùi hương nhẹ thomm phù hợp da da công dụng dư...,4,positive
1,75292434640,hài dịch vụ shipper sản phẩm mua chất tuyệt gi...,5,positive
2,13364343551,review nhẹ dòng sữa uống cân cm quan tạm grow ...,5,positive
3,84816190204,hiệu sạch thiết kế đẹp thiết kế gọn tiền lợi k...,5,positive
4,30102564756,giao hàng đóng gói đẹp toàn mùi pink review da...,4,positive


# Tokenize

TF-IDF is used for classical ML models. PhoBERT tokenization is used for transformer-based deep learning. This is where the ML and DL pipelines split: TF-IDF returns a sparse matrix, while PhoBERT tokenization returns PyTorch tensors.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer

def tokenize_tfidf(tokenizer, texts):
    return tokenizer.fit_transform(texts)

def phobert_tokenizer():
    return AutoTokenizer.from_pretrained("vinai/phobert-base")

def tokenize_phobert(tokenizer, text):
    return tokenizer(
        text,
        padding="max_length", # force all sequences to have the same length
        truncation=True, # truncate if token too long
        max_length=128,
        return_tensors="pt", # return pytorch tensor
    )

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tokens_ML = tokenize_tfidf(tfidf, df_baseline["segmented_review"])
tokens_ML

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 209902 stored elements and shape (10947, 5000)>

In [ ]:
phobert = phobert_tokenizer()
tokens_DL = df_lstm["segmented_review"].apply(
    lambda x: tokenize_phobert(phobert, x)
)
tokens_DL

0        {'input_ids': [[tensor(0), tensor(1602), tenso...
1        {'input_ids': [[tensor(0), tensor(70), tensor(...
2        {'input_ids': [[tensor(0), tensor(37973), tens...
3        {'input_ids': [[tensor(0), tensor(3256), tenso...
4        {'input_ids': [[tensor(0), tensor(574), tensor...
                               ...                        
10942    {'input_ids': [[tensor(0), tensor(68), tensor(...
10943    {'input_ids': [[tensor(0), tensor(2263), tenso...
10944    {'input_ids': [[tensor(0), tensor(17), tensor(...
10945    {'input_ids': [[tensor(0), tensor(7309), tenso...
10946    {'input_ids': [[tensor(0), tensor(2771), tenso...
Name: segmented_review, Length: 10947, dtype: object

In [ ]:
feature_names = tfidf.get_feature_names_out()

row = tokens_ML[0]

for idx, value in zip(row.indices, row.data):
    print(feature_names[idx], value)

mùi 0.037196988799985356
hương 0.07525651324295342
nhẹ 0.23871134211571687
phù_hợp 0.05658508991207095
da 0.1908793838437582
công_dụng 0.04781402794740319
dưỡng 0.15493108814424392
trắng 0.2096748928704023
ch 0.08795872612519028
thơm 0.07378861453804714
lợi 0.06709782073330335
khuyến 0.09294739651624691
thất 0.14145763894915975
gắng 0.1038004700398535
miêu_tả 0.09643060678464387
trải 0.11881542951722432
nghiệm 0.12094763933654812
tuần 0.06883141368554248
kết 0.07681386227150766
hộp 0.039477496408767535
body 0.08347296864546958
dove 0.10123007075796611
unilever 0.0897282995549618
hiện 0.0801571992133946
tai 0.06969527185540215
da_màu 0.10123007075796611
màu 0.04834666698742401
ngam 0.3963610721662092
um 0.09814101594489036
phan 0.13729421549849471
kbt 0.09422936969828631
tnao 0.1038004700398535
hiệu 0.060816720799153894
hình_ảnh 0.05140336926884341
day 0.18703211981005016
vọng 0.08715790027307439
chê 0.062177325545368214
nang 0.14643154387278406
cảm_giác 0.13157770544469835
bật 0.071278

In [ ]:
import os
import pickle
from scipy import sparse

# 1. Định nghĩa folder mới để lưu (Ví dụ: data/processed)
processed_dir = DATA_PROCESSED_DIR
os.makedirs(processed_dir, exist_ok=True)

# --- PHẦN 1: LƯU TOKENS CỦA MACHINE LEARNING (tokens_ML - Sparse Matrix) ---
# Cách tốt nhất để lưu ma trận thưa của Scipy là dùng hàm save_npz
ml_path = os.path.join(processed_dir, "tokens_ml_tfidf.npz")
sparse.save_npz(ml_path, tokens_ML)
print(f"Đã lưu ma trận TF-IDF tại: {ml_path}")


# --- PHẦN 2: LƯU TOKENS CỦA DEEP LEARNING (tokens_DL - Object/Tensor List) ---
# Vì tokens_DL chứa các dict/tensor phức tạp, dùng thư viện pickle hoặc torch.save là tối ưu nhất
dl_path = os.path.join(processed_dir, "tokens_dl_phobert.pkl")
with open(dl_path, "wb") as f:
    pickle.dump(tokens_DL, f)
print(f"Đã lưu tokens PhoBERT tại: {dl_path}")

<>:6: SyntaxWarning: invalid escape sequence '\T'
<>:6: SyntaxWarning: invalid escape sequence '\T'
C:\Users\Tony\AppData\Local\Temp\ipykernel_6212\3239077868.py:6: SyntaxWarning: invalid escape sequence '\T'
  processed_dir = "D:\Tony\Career\ProjectNhom\Customer_sentiment_analysis\data\processed"


Đã lưu ma trận TF-IDF tại: D:\Tony\Career\ProjectNhom\Customer_sentiment_analysis\data\processed\tokens_ml_tfidf.npz
Đã lưu tokens PhoBERT tại: D:\Tony\Career\ProjectNhom\Customer_sentiment_analysis\data\processed\tokens_dl_phobert.pkl
